In [ ]:
# Fabric Notebook: NB_POI_POSTGRES_SYNC_FINALIZE_V4_PROD_RECOVERY
# =============================================================================
# PIPELINE PARAMETER
# =============================================================================

SYNC_RUN_ID = ""


In [ ]:
# Fabric Notebook: NB_POI_POSTGRES_SYNC_FINALIZE_V4_PROD_RECOVERY
# =============================================================================
# PURPOSE
#   Finalize one PostgreSQL sync run AFTER all required Fabric Copy Data
#   activities have succeeded.
#
# PAIR WITH
#   NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY
#
# V3 CHANGE
#   Add poi_review and use an entity-specific sync key:
#
#       poi_master              -> poi_id
#       poi_detail              -> poi_id
#       poi_enrichment_display  -> poi_id
#       poi_review              -> review_id
#
# FLOW
#
#   PREPARE V3
#       |
#       +--> NO_DATA / copy_data_ready=false
#       |        -> END SUCCESS
#       |        -> DO NOT run FINALIZE
#       |
#       +--> SNAPSHOT_READY / copy_data_ready=true
#                |
#                +--> Copy poi_master
#                +--> Copy poi_detail
#                +--> Copy poi_enrichment_display
#                +--> Copy poi_review
#                          |
#                          +--> ALL 4 REQUIRED COPY DATA SUCCESS
#                                   |
#                                   v
#                              FINALIZE V3
#
# TOKEN-SAFE CLEANUP
#
#   For each entity:
#
#       shared.<sync_key>    = copy.<sync_key>
#       shared.change_token  = copy.change_token
#
#   Only the exact state that was snapshotted and successfully sent to
#   PostgreSQL is deleted from Shared STG.
#
# Example:
#
#   PREPARE:
#       copy_stg_poi_review
#       review_id = R001
#       token     = A
#
#   Producer updates R001 while Copy Data is running:
#       stg_poi_review_sync
#       review_id = R001
#       token     = B
#
#   FINALIZE:
#       A != B
#       -> current Shared row is NOT deleted
#       -> token B remains for the next sync
#
# IMPORTANT SAFETY RULES
#   1. Run ONLY after all required Copy Data activities succeed.
#   2. Pipeline dependencies into FINALIZE must be "Succeeded".
#   3. Sync pipeline concurrency MUST be 1.
#   4. COPY STG is immutable between PREPARE and FINALIZE.
#   5. COPY STG is truncated only after token-safe Shared STG cleanup succeeds.
#   6. If FINALIZE fails, remaining COPY STG is preserved for retry/debug.
#   7. Empty COPY STG for an entity is valid:
#        - PREPARE may have snapshotted 0 rows for that entity; or
#        - a previous FINALIZE retry already completed that entity.
#   8. PostgreSQL sink should use idempotent UPSERT:
#        poi_master/detail/enrichment -> poi_id
#        poi_review                   -> review_id
#
# PARAMETER
#   SYNC_RUN_ID:
#       Pass sync_run_id returned by PREPARE V3 when:
#           status = SNAPSHOT_READY or RECOVERY_READY
#           copy_data_ready = true
#
# NOTE
#   Mark the parameter cell below as a Fabric PARAMETERS cell when invoked
#   from a Fabric Pipeline Notebook activity.
# =============================================================================

from datetime import datetime, timezone
import json


# =============================================================================
# 0. PIPELINE PARAMETER
# =============================================================================

# SYNC_RUN_ID = ""


# =============================================================================
# 1. CONFIG
# =============================================================================

BRONZE_DB = "sgr_visitvn_dev.lh_vv_bronze.dbo"
GOLD_DB   = "sgr_visitvn_dev.lh_vv_gold.dbo"

PIPELINE_RUN_TBL = f"{BRONZE_DB}.brz_pipeline_run"

FLOW_NAME = "POI_POSTGRES_SYNC"
NOTEBOOK_NAME = "NB_POI_POSTGRES_SYNC_FINALIZE_V4_1_EXIT_SAFE"

ENTITY_CONFIG = {
    "poi_master": {
        "shared_staging": f"{GOLD_DB}.stg_poi_master_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_master",
        "sync_key": "poi_id",
    },
    "poi_detail": {
        "shared_staging": f"{GOLD_DB}.stg_poi_detail_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_detail",
        "sync_key": "poi_id",
    },
    "poi_enrichment_display": {
        "shared_staging": f"{GOLD_DB}.stg_poi_enrichment_display_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_enrichment_display",
        "sync_key": "poi_id",
    },
    "poi_review": {
        "shared_staging": f"{GOLD_DB}.stg_poi_review_sync",
        "copy_staging": f"{GOLD_DB}.copy_stg_poi_review",
        "sync_key": "review_id",
    },
}

REQUIRED_SHARED_META = {
    "change_token",
}

REQUIRED_COPY_META = {
    "change_token",
    "sync_run_id",
    "snapshot_at",
}

ALLOWED_PRE_FINALIZE_STATUSES = {
    "SNAPSHOT_READY",
    "RECOVERY_READY",
    "FINALIZING",
    "FINALIZE_FAILED",
}

started_at = datetime.now(timezone.utc)


# =============================================================================
# 2. HELPERS
# =============================================================================

def qcol(name: str) -> str:
    return f"`{name.replace('`', '``')}`"


def sql_string(value) -> str:
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def table_exists(table_name: str) -> bool:
    try:
        _ = spark.table(table_name).schema
        return True
    except Exception:
        return False


def table_column_set(table_name: str):
    return set(spark.table(table_name).columns)


def row_count(table_name: str) -> int:
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {table_name}
        """).first()["c"]
    )


def null_key_count(table_name: str, sync_key: str) -> int:
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {table_name}
            WHERE {qcol(sync_key)} IS NULL
        """).first()["c"]
    )


def duplicate_key_group_count(table_name: str, sync_key: str) -> int:
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM (
                SELECT {qcol(sync_key)}
                FROM {table_name}
                WHERE {qcol(sync_key)} IS NOT NULL
                GROUP BY {qcol(sync_key)}
                HAVING COUNT(*) > 1
            ) d
        """).first()["c"]
    )


def blank_change_token_count(table_name: str) -> int:
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {table_name}
            WHERE change_token IS NULL
               OR trim(change_token) = ''
        """).first()["c"]
    )


def validate_required_columns(
    table_name: str,
    required_columns,
    label: str,
):
    current = table_column_set(table_name)
    missing = sorted(set(required_columns) - current)

    if missing:
        raise RuntimeError(
            f"{label}_SCHEMA_MISMATCH "
            f"table={table_name} missing={missing}"
        )


def pipeline_run_columns():
    return table_column_set(PIPELINE_RUN_TBL)


def update_pipeline_run(
    status: str,
    error_message=None,
    close_run=False,
):
    if not table_exists(PIPELINE_RUN_TBL):
        return

    cols = pipeline_run_columns()
    assignments = []

    if "status" in cols:
        assignments.append(
            f"status = {sql_string(status)}"
        )

    if "current_notebook" in cols:
        assignments.append(
            f"current_notebook = {sql_string(NOTEBOOK_NAME)}"
        )

    if "error_message" in cols:
        msg = (
            str(error_message)[:4000]
            if error_message
            else None
        )
        assignments.append(
            f"error_message = {sql_string(msg)}"
        )

    if close_run and "ended_at" in cols:
        assignments.append(
            "ended_at = current_timestamp()"
        )

    if assignments:
        spark.sql(f"""
            UPDATE {PIPELINE_RUN_TBL}
            SET {", ".join(assignments)}
            WHERE run_id = {sql_string(SYNC_RUN_ID)}
        """)


def notebook_exit(payload: dict):
    result_json = json.dumps(
        payload,
        ensure_ascii=False,
        default=str,
    )

    print("=" * 100)
    print(result_json)
    print("=" * 100)

    try:
        notebookutils.notebook.exit(result_json)
    except NameError:
        pass


def count_copy_rows_for_run(
    copy_stg: str,
) -> int:
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {copy_stg}
            WHERE sync_run_id = {sql_string(SYNC_RUN_ID)}
        """).first()["c"]
    )


def count_exact_token_matches(
    shared: str,
    copy_stg: str,
    sync_key: str,
) -> int:
    """
    Current Shared state still exactly equals the snapshot sent to PostgreSQL.
    These rows are safe to delete.
    """
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {shared} s
            INNER JOIN {copy_stg} c
              ON s.{qcol(sync_key)} = c.{qcol(sync_key)}
             AND s.change_token = c.change_token
            WHERE c.sync_run_id = {sql_string(SYNC_RUN_ID)}
        """).first()["c"]
    )


def count_newer_shared_rows(
    shared: str,
    copy_stg: str,
    sync_key: str,
) -> int:
    """
    Same logical sync key exists in both tables, but Shared has a different
    change_token. This represents a newer producer state and MUST survive.
    """
    return int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {shared} s
            INNER JOIN {copy_stg} c
              ON s.{qcol(sync_key)} = c.{qcol(sync_key)}
            WHERE c.sync_run_id = {sql_string(SYNC_RUN_ID)}
              AND NOT (s.change_token <=> c.change_token)
        """).first()["c"]
    )


def validate_shared_staging(
    entity: str,
    cfg: dict,
):
    shared = cfg["shared_staging"]
    sync_key = cfg["sync_key"]

    if not table_exists(shared):
        raise RuntimeError(
            f"SHARED_STAGING_NOT_FOUND "
            f"entity={entity} table={shared}"
        )

    validate_required_columns(
        shared,
        REQUIRED_SHARED_META | {sync_key},
        "SHARED_STAGING",
    )

    null_keys = null_key_count(
        shared,
        sync_key,
    )

    if null_keys > 0:
        raise RuntimeError(
            f"SHARED_STAGING_NULL_SYNC_KEY "
            f"entity={entity} "
            f"sync_key={sync_key} "
            f"count={null_keys}"
        )

    duplicate_groups = duplicate_key_group_count(
        shared,
        sync_key,
    )

    if duplicate_groups > 0:
        raise RuntimeError(
            f"SHARED_STAGING_DUPLICATE_SYNC_KEY "
            f"entity={entity} "
            f"sync_key={sync_key} "
            f"duplicate_groups={duplicate_groups}"
        )

    blank_tokens = blank_change_token_count(
        shared
    )

    if blank_tokens > 0:
        raise RuntimeError(
            f"SHARED_STAGING_MISSING_CHANGE_TOKEN "
            f"entity={entity} "
            f"count={blank_tokens}"
        )


def validate_copy_staging(
    entity: str,
    cfg: dict,
):
    """
    Validate COPY STG ownership.

    Empty COPY STG is accepted for retry safety and for entities that had
    zero rows during PREPARE.

    If non-empty, EVERY row must belong to the current SYNC_RUN_ID.
    """
    copy_stg = cfg["copy_staging"]
    sync_key = cfg["sync_key"]

    if not table_exists(copy_stg):
        raise RuntimeError(
            f"COPY_STAGING_NOT_FOUND "
            f"entity={entity} table={copy_stg}"
        )

    validate_required_columns(
        copy_stg,
        REQUIRED_COPY_META | {sync_key},
        "COPY_STAGING",
    )

    total = row_count(
        copy_stg
    )

    if total == 0:
        return {
            "sync_key": sync_key,
            "row_count": 0,
            "status": "EMPTY_OR_ALREADY_FINALIZED",
        }

    null_keys = null_key_count(
        copy_stg,
        sync_key,
    )

    if null_keys > 0:
        raise RuntimeError(
            f"COPY_STAGING_NULL_SYNC_KEY "
            f"entity={entity} "
            f"sync_key={sync_key} "
            f"count={null_keys}"
        )

    duplicate_groups = duplicate_key_group_count(
        copy_stg,
        sync_key,
    )

    if duplicate_groups > 0:
        raise RuntimeError(
            f"COPY_STAGING_DUPLICATE_SYNC_KEY "
            f"entity={entity} "
            f"sync_key={sync_key} "
            f"duplicate_groups={duplicate_groups}"
        )

    blank_tokens = blank_change_token_count(
        copy_stg
    )

    if blank_tokens > 0:
        raise RuntimeError(
            f"COPY_STAGING_MISSING_CHANGE_TOKEN "
            f"entity={entity} "
            f"count={blank_tokens}"
        )

    invalid_run_count = int(
        spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {copy_stg}
            WHERE sync_run_id IS NULL
               OR sync_run_id <> {sql_string(SYNC_RUN_ID)}
        """).first()["c"]
    )

    if invalid_run_count > 0:
        actual_runs = [
            r["sync_run_id"]
            for r in spark.sql(f"""
                SELECT DISTINCT sync_run_id
                FROM {copy_stg}
                ORDER BY sync_run_id
            """).collect()
        ]

        raise RuntimeError(
            f"COPY_STG_RUN_ID_MISMATCH "
            f"entity={entity} "
            f"expected={SYNC_RUN_ID} "
            f"actual={actual_runs}"
        )

    owned_count = count_copy_rows_for_run(
        copy_stg
    )

    if owned_count != total:
        raise RuntimeError(
            f"COPY_STG_OWNERSHIP_COUNT_MISMATCH "
            f"entity={entity} "
            f"total={total} "
            f"owned={owned_count}"
        )

    return {
        "sync_key": sync_key,
        "row_count": total,
        "status": "READY_TO_FINALIZE",
    }


# =============================================================================
# 3. PARAMETER VALIDATION
# =============================================================================

SYNC_RUN_ID = str(
    SYNC_RUN_ID or ""
).strip()

if not SYNC_RUN_ID:
    raise RuntimeError(
        "SYNC_RUN_ID_REQUIRED. "
        "Pass sync_run_id returned by "
        "NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY."
    )

print(
    f"{NOTEBOOK_NAME} "
    f"sync_run_id={SYNC_RUN_ID}"
)


# =============================================================================
# 4. PREFLIGHT - AUDIT RUN
# =============================================================================

if not table_exists(PIPELINE_RUN_TBL):
    raise RuntimeError(
        f"PIPELINE_RUN_TABLE_NOT_FOUND "
        f"table={PIPELINE_RUN_TBL}"
    )

validate_required_columns(
    PIPELINE_RUN_TBL,
    {
        "run_id",
        "flow_name",
        "status",
        "started_at",
    },
    "PIPELINE_RUN",
)

run_row = spark.sql(f"""
    SELECT *
    FROM {PIPELINE_RUN_TBL}
    WHERE run_id = {sql_string(SYNC_RUN_ID)}
    LIMIT 1
""").first()

if run_row is None:
    raise RuntimeError(
        f"SYNC_RUN_NOT_FOUND "
        f"run_id={SYNC_RUN_ID}"
    )

run_dict = run_row.asDict(
    recursive=True
)

run_flow_name = run_dict.get(
    "flow_name"
)

if (
    run_flow_name is not None
    and run_flow_name != FLOW_NAME
):
    raise RuntimeError(
        f"SYNC_RUN_FLOW_MISMATCH "
        f"run_id={SYNC_RUN_ID} "
        f"expected={FLOW_NAME} "
        f"actual={run_flow_name}"
    )

run_status = run_dict.get(
    "status"
)


# =============================================================================
# 5. IDEMPOTENT ALREADY-SUCCESS PATH
# =============================================================================

if run_status == "SUCCESS":
    result = {
        "status": "SUCCESS",
        "sync_run_id": SYNC_RUN_ID,
        "flow_name": FLOW_NAME,
        "already_finalized": True,
        "copy_staging": {
            entity: {
                "table": cfg["copy_staging"],
                "sync_key": cfg["sync_key"],
                "row_count": (
                    row_count(cfg["copy_staging"])
                    if table_exists(cfg["copy_staging"])
                    else None
                ),
            }
            for entity, cfg
            in ENTITY_CONFIG.items()
        },
        "message": (
            "Sync run is already SUCCESS. "
            "No cleanup action executed."
        ),
    }

    notebook_exit(result)


# =============================================================================
# 6. VALIDATE RUN STATUS
# =============================================================================

if (
    run_status is not None
    and run_status not in ALLOWED_PRE_FINALIZE_STATUSES
):
    raise RuntimeError(
        f"SYNC_RUN_STATUS_NOT_FINALIZABLE "
        f"run_id={SYNC_RUN_ID} "
        f"status={run_status} "
        f"allowed={sorted(ALLOWED_PRE_FINALIZE_STATUSES)}"
    )


# =============================================================================
# 7. TABLE CONTRACT + INVARIANT PREFLIGHT
# =============================================================================

copy_state = {}

for entity, cfg in ENTITY_CONFIG.items():
    validate_shared_staging(
        entity,
        cfg,
    )

    copy_state[entity] = (
        validate_copy_staging(
            entity,
            cfg,
        )
    )

print(
    "FINALIZE PREFLIGHT = "
    + json.dumps(
        copy_state,
        ensure_ascii=False,
    )
)


# =============================================================================
# 8. MARK RUN AS FINALIZING
# =============================================================================
#
# This notebook intentionally does NOT inspect Copy Data activity status.
#
# Fabric Pipeline must call FINALIZE only after all four required Copy Data
# activities have Succeeded.
# =============================================================================

update_pipeline_run(
    status="FINALIZING",
    error_message=None,
    close_run=False,
)


# =============================================================================
# 9. TOKEN-SAFE CLEANUP
# =============================================================================

try:
    entity_results = {}

    for entity, cfg in ENTITY_CONFIG.items():
        shared = cfg["shared_staging"]
        copy_stg = cfg["copy_staging"]
        sync_key = cfg["sync_key"]

        copy_rows_before = row_count(
            copy_stg
        )

        # ---------------------------------------------------------------------
        # Retry-safe empty case.
        # ---------------------------------------------------------------------
        if copy_rows_before == 0:
            entity_results[entity] = {
                "sync_key": sync_key,
                "copy_rows": 0,
                "exact_token_matches_before": 0,
                "deleted_shared_rows": 0,
                "preserved_newer_rows": 0,
                "copy_rows_after": 0,
                "status": "EMPTY_OR_ALREADY_FINALIZED",
            }

            print(
                f"FINALIZE SKIP "
                f"entity={entity} "
                f"sync_key={sync_key}: "
                "COPY STG empty/already finalized"
            )

            continue

        # ---------------------------------------------------------------------
        # Measure current state.
        # ---------------------------------------------------------------------
        exact_matches_before = (
            count_exact_token_matches(
                shared,
                copy_stg,
                sync_key,
            )
        )

        newer_rows_before = (
            count_newer_shared_rows(
                shared,
                copy_stg,
                sync_key,
            )
        )

        print(
            f"FINALIZE BEFORE "
            f"entity={entity} "
            f"sync_key={sync_key} "
            f"copy_rows={copy_rows_before} "
            f"exact_matches={exact_matches_before} "
            f"newer_shared={newer_rows_before}"
        )

        # ---------------------------------------------------------------------
        # TOKEN-SAFE DELETE
        #
        # Only delete Shared state that still has the same token as the
        # immutable COPY snapshot owned by this sync run.
        # ---------------------------------------------------------------------
        spark.sql(f"""
            MERGE INTO {shared} s
            USING (
                SELECT
                    {qcol(sync_key)} AS sync_key_value,
                    change_token
                FROM {copy_stg}
                WHERE sync_run_id = {sql_string(SYNC_RUN_ID)}
            ) c
            ON s.{qcol(sync_key)} = c.sync_key_value
           AND s.change_token = c.change_token

            WHEN MATCHED THEN DELETE
        """)

        # ---------------------------------------------------------------------
        # Exact snapshot states must now be gone.
        # ---------------------------------------------------------------------
        exact_matches_after = (
            count_exact_token_matches(
                shared,
                copy_stg,
                sync_key,
            )
        )

        if exact_matches_after != 0:
            raise RuntimeError(
                f"SHARED_STAGING_CLEANUP_INCOMPLETE "
                f"entity={entity} "
                f"sync_key={sync_key} "
                f"remaining_exact_matches={exact_matches_after}"
            )

        # ---------------------------------------------------------------------
        # Newer producer states must remain.
        # ---------------------------------------------------------------------
        newer_rows_after = (
            count_newer_shared_rows(
                shared,
                copy_stg,
                sync_key,
            )
        )

        if newer_rows_after < newer_rows_before:
            raise RuntimeError(
                f"NEWER_SHARED_STATE_LOST "
                f"entity={entity} "
                f"sync_key={sync_key} "
                f"before={newer_rows_before} "
                f"after={newer_rows_after}"
            )

        # ---------------------------------------------------------------------
        # Only after Shared cleanup succeeds, truncate this COPY STG.
        # ---------------------------------------------------------------------
        spark.sql(
            f"TRUNCATE TABLE {copy_stg}"
        )

        copy_rows_after = row_count(
            copy_stg
        )

        if copy_rows_after != 0:
            raise RuntimeError(
                f"COPY_STG_TRUNCATE_FAILED "
                f"entity={entity} "
                f"remaining_rows={copy_rows_after}"
            )

        entity_results[entity] = {
            "sync_key": sync_key,
            "copy_rows": copy_rows_before,
            "exact_token_matches_before": exact_matches_before,
            "deleted_shared_rows": exact_matches_before,
            "preserved_newer_rows": newer_rows_after,
            "copy_rows_after": copy_rows_after,
            "status": "FINALIZED",
        }

        print(
            f"FINALIZED "
            f"entity={entity} "
            f"sync_key={sync_key} "
            f"copy_rows={copy_rows_before} "
            f"deleted_shared={exact_matches_before} "
            f"preserved_newer={newer_rows_after}"
        )


    # =========================================================================
    # 10. GLOBAL POST-FINALIZE VALIDATION
    # =========================================================================

    remaining_copy = {
        entity: row_count(
            cfg["copy_staging"]
        )
        for entity, cfg
        in ENTITY_CONFIG.items()
    }

    non_empty_remaining = {
        entity: count
        for entity, count
        in remaining_copy.items()
        if count > 0
    }

    if non_empty_remaining:
        raise RuntimeError(
            f"FINAL_COPY_STG_NOT_EMPTY "
            f"counts={non_empty_remaining}"
        )

    # Shared uniqueness invariant:
    #   first three entities -> unique poi_id
    #   poi_review           -> unique review_id
    for entity, cfg in ENTITY_CONFIG.items():
        shared = cfg["shared_staging"]
        sync_key = cfg["sync_key"]

        null_keys = null_key_count(
            shared,
            sync_key,
        )

        if null_keys > 0:
            raise RuntimeError(
                f"POST_FINALIZE_SHARED_NULL_SYNC_KEY "
                f"entity={entity} "
                f"sync_key={sync_key} "
                f"count={null_keys}"
            )

        duplicate_groups = duplicate_key_group_count(
            shared,
            sync_key,
        )

        if duplicate_groups > 0:
            raise RuntimeError(
                f"POST_FINALIZE_SHARED_DUPLICATE_SYNC_KEY "
                f"entity={entity} "
                f"sync_key={sync_key} "
                f"duplicate_groups={duplicate_groups}"
            )

        blank_tokens = blank_change_token_count(
            shared
        )

        if blank_tokens > 0:
            raise RuntimeError(
                f"POST_FINALIZE_SHARED_MISSING_CHANGE_TOKEN "
                f"entity={entity} "
                f"count={blank_tokens}"
            )


    # =========================================================================
    # 11. CLOSE SYNC RUN
    # =========================================================================

    update_pipeline_run(
        status="SUCCESS",
        error_message=None,
        close_run=True,
    )

    ended_at = datetime.now(
        timezone.utc
    )

    result = {
        "status": "SUCCESS",
        "sync_run_id": SYNC_RUN_ID,
        "flow_name": FLOW_NAME,
        "already_finalized": False,
        "started_finalize_at": started_at.isoformat(),
        "ended_finalize_at": ended_at.isoformat(),
        "entities": entity_results,
        "copy_staging_empty": True,
        "next_sync_allowed": True,
    }

    # Do NOT call notebook_exit here. Fabric notebook exit is control-flow
    # and must not be caught by the failure handler below.


# =============================================================================
# 12. FAILURE PATH
# =============================================================================

except Exception as e:
    error_message = str(e)

    # -------------------------------------------------------------------------
    # DO NOT truncate remaining COPY STG here.
    #
    # Retry behavior:
    #   - entities already finalized may have empty COPY STG;
    #   - remaining entities retain COPY STG;
    #   - retry accepts empty/already-finalized entities and continues.
    #
    # This makes FINALIZE retry-safe at entity level.
    # -------------------------------------------------------------------------
    try:
        update_pipeline_run(
            status="FINALIZE_FAILED",
            error_message=error_message,
            close_run=False,
        )
    except Exception as audit_error:
        print(
            "WARNING: failed to update "
            f"sync audit: {audit_error}"
        )

    raise

# Deliberately outside cleanup try/except so a successful notebook exit cannot
# be reclassified as FINALIZE_FAILED.
notebook_exit(result)


In [ ]:
# # Fabric Notebook: NB_POI_POSTGRES_SYNC_FINALIZE_V4_PROD_RECOVERY
# # =============================================================================
# # PURPOSE
# #   Finalize one PostgreSQL sync run AFTER all required Fabric Copy Data
# #   activities have succeeded.
# #
# # PAIR WITH
# #   NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY
# #
# # V3 CHANGE
# #   Add poi_review and use an entity-specific sync key:
# #
# #       poi_master              -> poi_id
# #       poi_detail              -> poi_id
# #       poi_enrichment_display  -> poi_id
# #       poi_review              -> review_id
# #
# # FLOW
# #
# #   PREPARE V3
# #       |
# #       +--> NO_DATA / copy_data_ready=false
# #       |        -> END SUCCESS
# #       |        -> DO NOT run FINALIZE
# #       |
# #       +--> SNAPSHOT_READY / copy_data_ready=true
# #                |
# #                +--> Copy poi_master
# #                +--> Copy poi_detail
# #                +--> Copy poi_enrichment_display
# #                +--> Copy poi_review
# #                          |
# #                          +--> ALL 4 REQUIRED COPY DATA SUCCESS
# #                                   |
# #                                   v
# #                              FINALIZE V3
# #
# # TOKEN-SAFE CLEANUP
# #
# #   For each entity:
# #
# #       shared.<sync_key>    = copy.<sync_key>
# #       shared.change_token  = copy.change_token
# #
# #   Only the exact state that was snapshotted and successfully sent to
# #   PostgreSQL is deleted from Shared STG.
# #
# # Example:
# #
# #   PREPARE:
# #       copy_stg_poi_review
# #       review_id = R001
# #       token     = A
# #
# #   Producer updates R001 while Copy Data is running:
# #       stg_poi_review_sync
# #       review_id = R001
# #       token     = B
# #
# #   FINALIZE:
# #       A != B
# #       -> current Shared row is NOT deleted
# #       -> token B remains for the next sync
# #
# # IMPORTANT SAFETY RULES
# #   1. Run ONLY after all required Copy Data activities succeed.
# #   2. Pipeline dependencies into FINALIZE must be "Succeeded".
# #   3. Sync pipeline concurrency MUST be 1.
# #   4. COPY STG is immutable between PREPARE and FINALIZE.
# #   5. COPY STG is truncated only after token-safe Shared STG cleanup succeeds.
# #   6. If FINALIZE fails, remaining COPY STG is preserved for retry/debug.
# #   7. Empty COPY STG for an entity is valid:
# #        - PREPARE may have snapshotted 0 rows for that entity; or
# #        - a previous FINALIZE retry already completed that entity.
# #   8. PostgreSQL sink should use idempotent UPSERT:
# #        poi_master/detail/enrichment -> poi_id
# #        poi_review                   -> review_id
# #
# # PARAMETER
# #   SYNC_RUN_ID:
# #       Pass sync_run_id returned by PREPARE V3 when:
# #           status = SNAPSHOT_READY or RECOVERY_READY
# #           copy_data_ready = true
# #
# # NOTE
# #   Mark the parameter cell below as a Fabric PARAMETERS cell when invoked
# #   from a Fabric Pipeline Notebook activity.
# # =============================================================================

# from datetime import datetime, timezone
# import json


# # =============================================================================
# # 0. PIPELINE PARAMETER
# # =============================================================================

# #SYNC_RUN_ID = ""


# # =============================================================================
# # 1. CONFIG
# # =============================================================================

# BRONZE_DB = "sgr_visitvn_dev.lh_vv_bronze.dbo"
# GOLD_DB   = "sgr_visitvn_dev.lh_vv_gold.dbo"

# PIPELINE_RUN_TBL = f"{BRONZE_DB}.brz_pipeline_run"

# FLOW_NAME = "POI_POSTGRES_SYNC"
# NOTEBOOK_NAME = "NB_POI_POSTGRES_SYNC_FINALIZE_V4_PROD_RECOVERY"

# ENTITY_CONFIG = {
#     "poi_master": {
#         "shared_staging": f"{GOLD_DB}.stg_poi_master_sync",
#         "copy_staging": f"{GOLD_DB}.copy_stg_poi_master",
#         "sync_key": "poi_id",
#     },
#     "poi_detail": {
#         "shared_staging": f"{GOLD_DB}.stg_poi_detail_sync",
#         "copy_staging": f"{GOLD_DB}.copy_stg_poi_detail",
#         "sync_key": "poi_id",
#     },
#     "poi_enrichment_display": {
#         "shared_staging": f"{GOLD_DB}.stg_poi_enrichment_display_sync",
#         "copy_staging": f"{GOLD_DB}.copy_stg_poi_enrichment_display",
#         "sync_key": "poi_id",
#     },
#     "poi_review": {
#         "shared_staging": f"{GOLD_DB}.stg_poi_review_sync",
#         "copy_staging": f"{GOLD_DB}.copy_stg_poi_review",
#         "sync_key": "review_id",
#     },
# }

# REQUIRED_SHARED_META = {
#     "change_token",
# }

# REQUIRED_COPY_META = {
#     "change_token",
#     "sync_run_id",
#     "snapshot_at",
# }

# ALLOWED_PRE_FINALIZE_STATUSES = {
#     "SNAPSHOT_READY",
#     "RECOVERY_READY",
#     "FINALIZING",
#     "FINALIZE_FAILED",
# }

# started_at = datetime.now(timezone.utc)


# # =============================================================================
# # 2. HELPERS
# # =============================================================================

# def qcol(name: str) -> str:
#     return f"`{name.replace('`', '``')}`"


# def sql_string(value) -> str:
#     if value is None:
#         return "NULL"
#     return "'" + str(value).replace("'", "''") + "'"


# def table_exists(table_name: str) -> bool:
#     try:
#         _ = spark.table(table_name).schema
#         return True
#     except Exception:
#         return False


# def table_column_set(table_name: str):
#     return set(spark.table(table_name).columns)


# def row_count(table_name: str) -> int:
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {table_name}
#         """).first()["c"]
#     )


# def null_key_count(table_name: str, sync_key: str) -> int:
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {table_name}
#             WHERE {qcol(sync_key)} IS NULL
#         """).first()["c"]
#     )


# def duplicate_key_group_count(table_name: str, sync_key: str) -> int:
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM (
#                 SELECT {qcol(sync_key)}
#                 FROM {table_name}
#                 WHERE {qcol(sync_key)} IS NOT NULL
#                 GROUP BY {qcol(sync_key)}
#                 HAVING COUNT(*) > 1
#             ) d
#         """).first()["c"]
#     )


# def blank_change_token_count(table_name: str) -> int:
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {table_name}
#             WHERE change_token IS NULL
#                OR trim(change_token) = ''
#         """).first()["c"]
#     )


# def validate_required_columns(
#     table_name: str,
#     required_columns,
#     label: str,
# ):
#     current = table_column_set(table_name)
#     missing = sorted(set(required_columns) - current)

#     if missing:
#         raise RuntimeError(
#             f"{label}_SCHEMA_MISMATCH "
#             f"table={table_name} missing={missing}"
#         )


# def pipeline_run_columns():
#     return table_column_set(PIPELINE_RUN_TBL)


# def update_pipeline_run(
#     status: str,
#     error_message=None,
#     close_run=False,
# ):
#     if not table_exists(PIPELINE_RUN_TBL):
#         return

#     cols = pipeline_run_columns()
#     assignments = []

#     if "status" in cols:
#         assignments.append(
#             f"status = {sql_string(status)}"
#         )

#     if "current_notebook" in cols:
#         assignments.append(
#             f"current_notebook = {sql_string(NOTEBOOK_NAME)}"
#         )

#     if "error_message" in cols:
#         msg = (
#             str(error_message)[:4000]
#             if error_message
#             else None
#         )
#         assignments.append(
#             f"error_message = {sql_string(msg)}"
#         )

#     if close_run and "ended_at" in cols:
#         assignments.append(
#             "ended_at = current_timestamp()"
#         )

#     if assignments:
#         spark.sql(f"""
#             UPDATE {PIPELINE_RUN_TBL}
#             SET {", ".join(assignments)}
#             WHERE run_id = {sql_string(SYNC_RUN_ID)}
#         """)


# def notebook_exit(payload: dict):
#     result_json = json.dumps(
#         payload,
#         ensure_ascii=False,
#         default=str,
#     )

#     print("=" * 100)
#     print(result_json)
#     print("=" * 100)

#     try:
#         notebookutils.notebook.exit(result_json)
#     except NameError:
#         pass


# def count_copy_rows_for_run(
#     copy_stg: str,
# ) -> int:
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {copy_stg}
#             WHERE sync_run_id = {sql_string(SYNC_RUN_ID)}
#         """).first()["c"]
#     )


# def count_exact_token_matches(
#     shared: str,
#     copy_stg: str,
#     sync_key: str,
# ) -> int:
#     """
#     Current Shared state still exactly equals the snapshot sent to PostgreSQL.
#     These rows are safe to delete.
#     """
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {shared} s
#             INNER JOIN {copy_stg} c
#               ON s.{qcol(sync_key)} = c.{qcol(sync_key)}
#              AND s.change_token = c.change_token
#             WHERE c.sync_run_id = {sql_string(SYNC_RUN_ID)}
#         """).first()["c"]
#     )


# def count_newer_shared_rows(
#     shared: str,
#     copy_stg: str,
#     sync_key: str,
# ) -> int:
#     """
#     Same logical sync key exists in both tables, but Shared has a different
#     change_token. This represents a newer producer state and MUST survive.
#     """
#     return int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {shared} s
#             INNER JOIN {copy_stg} c
#               ON s.{qcol(sync_key)} = c.{qcol(sync_key)}
#             WHERE c.sync_run_id = {sql_string(SYNC_RUN_ID)}
#               AND NOT (s.change_token <=> c.change_token)
#         """).first()["c"]
#     )


# def validate_shared_staging(
#     entity: str,
#     cfg: dict,
# ):
#     shared = cfg["shared_staging"]
#     sync_key = cfg["sync_key"]

#     if not table_exists(shared):
#         raise RuntimeError(
#             f"SHARED_STAGING_NOT_FOUND "
#             f"entity={entity} table={shared}"
#         )

#     validate_required_columns(
#         shared,
#         REQUIRED_SHARED_META | {sync_key},
#         "SHARED_STAGING",
#     )

#     null_keys = null_key_count(
#         shared,
#         sync_key,
#     )

#     if null_keys > 0:
#         raise RuntimeError(
#             f"SHARED_STAGING_NULL_SYNC_KEY "
#             f"entity={entity} "
#             f"sync_key={sync_key} "
#             f"count={null_keys}"
#         )

#     duplicate_groups = duplicate_key_group_count(
#         shared,
#         sync_key,
#     )

#     if duplicate_groups > 0:
#         raise RuntimeError(
#             f"SHARED_STAGING_DUPLICATE_SYNC_KEY "
#             f"entity={entity} "
#             f"sync_key={sync_key} "
#             f"duplicate_groups={duplicate_groups}"
#         )

#     blank_tokens = blank_change_token_count(
#         shared
#     )

#     if blank_tokens > 0:
#         raise RuntimeError(
#             f"SHARED_STAGING_MISSING_CHANGE_TOKEN "
#             f"entity={entity} "
#             f"count={blank_tokens}"
#         )


# def validate_copy_staging(
#     entity: str,
#     cfg: dict,
# ):
#     """
#     Validate COPY STG ownership.

#     Empty COPY STG is accepted for retry safety and for entities that had
#     zero rows during PREPARE.

#     If non-empty, EVERY row must belong to the current SYNC_RUN_ID.
#     """
#     copy_stg = cfg["copy_staging"]
#     sync_key = cfg["sync_key"]

#     if not table_exists(copy_stg):
#         raise RuntimeError(
#             f"COPY_STAGING_NOT_FOUND "
#             f"entity={entity} table={copy_stg}"
#         )

#     validate_required_columns(
#         copy_stg,
#         REQUIRED_COPY_META | {sync_key},
#         "COPY_STAGING",
#     )

#     total = row_count(
#         copy_stg
#     )

#     if total == 0:
#         return {
#             "sync_key": sync_key,
#             "row_count": 0,
#             "status": "EMPTY_OR_ALREADY_FINALIZED",
#         }

#     null_keys = null_key_count(
#         copy_stg,
#         sync_key,
#     )

#     if null_keys > 0:
#         raise RuntimeError(
#             f"COPY_STAGING_NULL_SYNC_KEY "
#             f"entity={entity} "
#             f"sync_key={sync_key} "
#             f"count={null_keys}"
#         )

#     duplicate_groups = duplicate_key_group_count(
#         copy_stg,
#         sync_key,
#     )

#     if duplicate_groups > 0:
#         raise RuntimeError(
#             f"COPY_STAGING_DUPLICATE_SYNC_KEY "
#             f"entity={entity} "
#             f"sync_key={sync_key} "
#             f"duplicate_groups={duplicate_groups}"
#         )

#     blank_tokens = blank_change_token_count(
#         copy_stg
#     )

#     if blank_tokens > 0:
#         raise RuntimeError(
#             f"COPY_STAGING_MISSING_CHANGE_TOKEN "
#             f"entity={entity} "
#             f"count={blank_tokens}"
#         )

#     invalid_run_count = int(
#         spark.sql(f"""
#             SELECT COUNT(*) AS c
#             FROM {copy_stg}
#             WHERE sync_run_id IS NULL
#                OR sync_run_id <> {sql_string(SYNC_RUN_ID)}
#         """).first()["c"]
#     )

#     if invalid_run_count > 0:
#         actual_runs = [
#             r["sync_run_id"]
#             for r in spark.sql(f"""
#                 SELECT DISTINCT sync_run_id
#                 FROM {copy_stg}
#                 ORDER BY sync_run_id
#             """).collect()
#         ]

#         raise RuntimeError(
#             f"COPY_STG_RUN_ID_MISMATCH "
#             f"entity={entity} "
#             f"expected={SYNC_RUN_ID} "
#             f"actual={actual_runs}"
#         )

#     owned_count = count_copy_rows_for_run(
#         copy_stg
#     )

#     if owned_count != total:
#         raise RuntimeError(
#             f"COPY_STG_OWNERSHIP_COUNT_MISMATCH "
#             f"entity={entity} "
#             f"total={total} "
#             f"owned={owned_count}"
#         )

#     return {
#         "sync_key": sync_key,
#         "row_count": total,
#         "status": "READY_TO_FINALIZE",
#     }


# # =============================================================================
# # 3. PARAMETER VALIDATION
# # =============================================================================

# SYNC_RUN_ID = str(
#     SYNC_RUN_ID or ""
# ).strip()

# if not SYNC_RUN_ID:
#     raise RuntimeError(
#         "SYNC_RUN_ID_REQUIRED. "
#         "Pass sync_run_id returned by "
#         "NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY."
#     )

# print(
#     f"{NOTEBOOK_NAME} "
#     f"sync_run_id={SYNC_RUN_ID}"
# )


# # =============================================================================
# # 4. PREFLIGHT - AUDIT RUN
# # =============================================================================

# if not table_exists(PIPELINE_RUN_TBL):
#     raise RuntimeError(
#         f"PIPELINE_RUN_TABLE_NOT_FOUND "
#         f"table={PIPELINE_RUN_TBL}"
#     )

# validate_required_columns(
#     PIPELINE_RUN_TBL,
#     {
#         "run_id",
#         "flow_name",
#         "status",
#         "started_at",
#     },
#     "PIPELINE_RUN",
# )

# run_row = spark.sql(f"""
#     SELECT *
#     FROM {PIPELINE_RUN_TBL}
#     WHERE run_id = {sql_string(SYNC_RUN_ID)}
#     LIMIT 1
# """).first()

# if run_row is None:
#     raise RuntimeError(
#         f"SYNC_RUN_NOT_FOUND "
#         f"run_id={SYNC_RUN_ID}"
#     )

# run_dict = run_row.asDict(
#     recursive=True
# )

# run_flow_name = run_dict.get(
#     "flow_name"
# )

# if (
#     run_flow_name is not None
#     and run_flow_name != FLOW_NAME
# ):
#     raise RuntimeError(
#         f"SYNC_RUN_FLOW_MISMATCH "
#         f"run_id={SYNC_RUN_ID} "
#         f"expected={FLOW_NAME} "
#         f"actual={run_flow_name}"
#     )

# run_status = run_dict.get(
#     "status"
# )


# # =============================================================================
# # 5. IDEMPOTENT ALREADY-SUCCESS PATH
# # =============================================================================

# if run_status == "SUCCESS":
#     result = {
#         "status": "SUCCESS",
#         "sync_run_id": SYNC_RUN_ID,
#         "flow_name": FLOW_NAME,
#         "already_finalized": True,
#         "copy_staging": {
#             entity: {
#                 "table": cfg["copy_staging"],
#                 "sync_key": cfg["sync_key"],
#                 "row_count": (
#                     row_count(cfg["copy_staging"])
#                     if table_exists(cfg["copy_staging"])
#                     else None
#                 ),
#             }
#             for entity, cfg
#             in ENTITY_CONFIG.items()
#         },
#         "message": (
#             "Sync run is already SUCCESS. "
#             "No cleanup action executed."
#         ),
#     }

#     notebook_exit(result)


# # =============================================================================
# # 6. VALIDATE RUN STATUS
# # =============================================================================

# if (
#     run_status is not None
#     and run_status not in ALLOWED_PRE_FINALIZE_STATUSES
# ):
#     raise RuntimeError(
#         f"SYNC_RUN_STATUS_NOT_FINALIZABLE "
#         f"run_id={SYNC_RUN_ID} "
#         f"status={run_status} "
#         f"allowed={sorted(ALLOWED_PRE_FINALIZE_STATUSES)}"
#     )


# # =============================================================================
# # 7. TABLE CONTRACT + INVARIANT PREFLIGHT
# # =============================================================================

# copy_state = {}

# for entity, cfg in ENTITY_CONFIG.items():
#     validate_shared_staging(
#         entity,
#         cfg,
#     )

#     copy_state[entity] = (
#         validate_copy_staging(
#             entity,
#             cfg,
#         )
#     )

# print(
#     "FINALIZE PREFLIGHT = "
#     + json.dumps(
#         copy_state,
#         ensure_ascii=False,
#     )
# )


# # =============================================================================
# # 8. MARK RUN AS FINALIZING
# # =============================================================================
# #
# # This notebook intentionally does NOT inspect Copy Data activity status.
# #
# # Fabric Pipeline must call FINALIZE only after all four required Copy Data
# # activities have Succeeded.
# # =============================================================================

# update_pipeline_run(
#     status="FINALIZING",
#     error_message=None,
#     close_run=False,
# )


# # =============================================================================
# # 9. TOKEN-SAFE CLEANUP
# # =============================================================================

# try:
#     entity_results = {}

#     for entity, cfg in ENTITY_CONFIG.items():
#         shared = cfg["shared_staging"]
#         copy_stg = cfg["copy_staging"]
#         sync_key = cfg["sync_key"]

#         copy_rows_before = row_count(
#             copy_stg
#         )

#         # ---------------------------------------------------------------------
#         # Retry-safe empty case.
#         # ---------------------------------------------------------------------
#         if copy_rows_before == 0:
#             entity_results[entity] = {
#                 "sync_key": sync_key,
#                 "copy_rows": 0,
#                 "exact_token_matches_before": 0,
#                 "deleted_shared_rows": 0,
#                 "preserved_newer_rows": 0,
#                 "copy_rows_after": 0,
#                 "status": "EMPTY_OR_ALREADY_FINALIZED",
#             }

#             print(
#                 f"FINALIZE SKIP "
#                 f"entity={entity} "
#                 f"sync_key={sync_key}: "
#                 "COPY STG empty/already finalized"
#             )

#             continue

#         # ---------------------------------------------------------------------
#         # Measure current state.
#         # ---------------------------------------------------------------------
#         exact_matches_before = (
#             count_exact_token_matches(
#                 shared,
#                 copy_stg,
#                 sync_key,
#             )
#         )

#         newer_rows_before = (
#             count_newer_shared_rows(
#                 shared,
#                 copy_stg,
#                 sync_key,
#             )
#         )

#         print(
#             f"FINALIZE BEFORE "
#             f"entity={entity} "
#             f"sync_key={sync_key} "
#             f"copy_rows={copy_rows_before} "
#             f"exact_matches={exact_matches_before} "
#             f"newer_shared={newer_rows_before}"
#         )

#         # ---------------------------------------------------------------------
#         # TOKEN-SAFE DELETE
#         #
#         # Only delete Shared state that still has the same token as the
#         # immutable COPY snapshot owned by this sync run.
#         # ---------------------------------------------------------------------
#         spark.sql(f"""
#             MERGE INTO {shared} s
#             USING (
#                 SELECT
#                     {qcol(sync_key)} AS sync_key_value,
#                     change_token
#                 FROM {copy_stg}
#                 WHERE sync_run_id = {sql_string(SYNC_RUN_ID)}
#             ) c
#             ON s.{qcol(sync_key)} = c.sync_key_value
#            AND s.change_token = c.change_token

#             WHEN MATCHED THEN DELETE
#         """)

#         # ---------------------------------------------------------------------
#         # Exact snapshot states must now be gone.
#         # ---------------------------------------------------------------------
#         exact_matches_after = (
#             count_exact_token_matches(
#                 shared,
#                 copy_stg,
#                 sync_key,
#             )
#         )

#         if exact_matches_after != 0:
#             raise RuntimeError(
#                 f"SHARED_STAGING_CLEANUP_INCOMPLETE "
#                 f"entity={entity} "
#                 f"sync_key={sync_key} "
#                 f"remaining_exact_matches={exact_matches_after}"
#             )

#         # ---------------------------------------------------------------------
#         # Newer producer states must remain.
#         # ---------------------------------------------------------------------
#         newer_rows_after = (
#             count_newer_shared_rows(
#                 shared,
#                 copy_stg,
#                 sync_key,
#             )
#         )

#         if newer_rows_after < newer_rows_before:
#             raise RuntimeError(
#                 f"NEWER_SHARED_STATE_LOST "
#                 f"entity={entity} "
#                 f"sync_key={sync_key} "
#                 f"before={newer_rows_before} "
#                 f"after={newer_rows_after}"
#             )

#         # ---------------------------------------------------------------------
#         # Only after Shared cleanup succeeds, truncate this COPY STG.
#         # ---------------------------------------------------------------------
#         spark.sql(
#             f"TRUNCATE TABLE {copy_stg}"
#         )

#         copy_rows_after = row_count(
#             copy_stg
#         )

#         if copy_rows_after != 0:
#             raise RuntimeError(
#                 f"COPY_STG_TRUNCATE_FAILED "
#                 f"entity={entity} "
#                 f"remaining_rows={copy_rows_after}"
#             )

#         entity_results[entity] = {
#             "sync_key": sync_key,
#             "copy_rows": copy_rows_before,
#             "exact_token_matches_before": exact_matches_before,
#             "deleted_shared_rows": exact_matches_before,
#             "preserved_newer_rows": newer_rows_after,
#             "copy_rows_after": copy_rows_after,
#             "status": "FINALIZED",
#         }

#         print(
#             f"FINALIZED "
#             f"entity={entity} "
#             f"sync_key={sync_key} "
#             f"copy_rows={copy_rows_before} "
#             f"deleted_shared={exact_matches_before} "
#             f"preserved_newer={newer_rows_after}"
#         )


#     # =========================================================================
#     # 10. GLOBAL POST-FINALIZE VALIDATION
#     # =========================================================================

#     remaining_copy = {
#         entity: row_count(
#             cfg["copy_staging"]
#         )
#         for entity, cfg
#         in ENTITY_CONFIG.items()
#     }

#     non_empty_remaining = {
#         entity: count
#         for entity, count
#         in remaining_copy.items()
#         if count > 0
#     }

#     if non_empty_remaining:
#         raise RuntimeError(
#             f"FINAL_COPY_STG_NOT_EMPTY "
#             f"counts={non_empty_remaining}"
#         )

#     # Shared uniqueness invariant:
#     #   first three entities -> unique poi_id
#     #   poi_review           -> unique review_id
#     for entity, cfg in ENTITY_CONFIG.items():
#         shared = cfg["shared_staging"]
#         sync_key = cfg["sync_key"]

#         null_keys = null_key_count(
#             shared,
#             sync_key,
#         )

#         if null_keys > 0:
#             raise RuntimeError(
#                 f"POST_FINALIZE_SHARED_NULL_SYNC_KEY "
#                 f"entity={entity} "
#                 f"sync_key={sync_key} "
#                 f"count={null_keys}"
#             )

#         duplicate_groups = duplicate_key_group_count(
#             shared,
#             sync_key,
#         )

#         if duplicate_groups > 0:
#             raise RuntimeError(
#                 f"POST_FINALIZE_SHARED_DUPLICATE_SYNC_KEY "
#                 f"entity={entity} "
#                 f"sync_key={sync_key} "
#                 f"duplicate_groups={duplicate_groups}"
#             )

#         blank_tokens = blank_change_token_count(
#             shared
#         )

#         if blank_tokens > 0:
#             raise RuntimeError(
#                 f"POST_FINALIZE_SHARED_MISSING_CHANGE_TOKEN "
#                 f"entity={entity} "
#                 f"count={blank_tokens}"
#             )


#     # =========================================================================
#     # 11. CLOSE SYNC RUN
#     # =========================================================================

#     update_pipeline_run(
#         status="SUCCESS",
#         error_message=None,
#         close_run=True,
#     )

#     ended_at = datetime.now(
#         timezone.utc
#     )

#     result = {
#         "status": "SUCCESS",
#         "sync_run_id": SYNC_RUN_ID,
#         "flow_name": FLOW_NAME,
#         "already_finalized": False,
#         "started_finalize_at": started_at.isoformat(),
#         "ended_finalize_at": ended_at.isoformat(),
#         "entities": entity_results,
#         "copy_staging_empty": True,
#         "next_sync_allowed": True,
#     }

#     notebook_exit(result)


# # =============================================================================
# # 12. FAILURE PATH
# # =============================================================================

# except Exception as e:
#     error_message = str(e)

#     # -------------------------------------------------------------------------
#     # DO NOT truncate remaining COPY STG here.
#     #
#     # Retry behavior:
#     #   - entities already finalized may have empty COPY STG;
#     #   - remaining entities retain COPY STG;
#     #   - retry accepts empty/already-finalized entities and continues.
#     #
#     # This makes FINALIZE retry-safe at entity level.
#     # -------------------------------------------------------------------------
#     try:
#         update_pipeline_run(
#             status="FINALIZE_FAILED",
#             error_message=error_message,
#             close_run=False,
#         )
#     except Exception as audit_error:
#         print(
#             "WARNING: failed to update "
#             f"sync audit: {audit_error}"
#         )

#     raise
